# Custom HuggingFace model: serve and evaluate on Modal

**What this tutorial teaches.** How to use a model that isn't in
the built-in catalog. The built-in classes (`Qwen3_4B`, `Qwen3_0_6B`,
etc.) are concrete `HFModelConfiguration` subclasses, but nothing
stops you from writing your own inline — no registry, no library
fork, just a subclass in your tutorial file.

**What it actually runs.** We serve
[`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)
(a 135M-param Llama-family model) via vLLM and run a small
GSM8K math eval against it. No training — the goal is to prove
the custom-model seam works end-to-end with serve + eval,
cheaply.

**Prerequisites.** A Modal account with a `huggingface-secret`
containing your `HF_TOKEN`.

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from typing import Any

from modal_training_gym.common.dataset import HuggingFaceDataset
from modal_training_gym.common.eval import EvalConfig, EvalRowResult
from modal_training_gym.common.models import HFModelConfiguration

## Define the custom model

Subclass `HFModelConfiguration` with just `model_name` — the
base class already implements `download()` via
`huggingface_hub.snapshot_download`. That's the whole seam.

For models that need post-download conversion or special
tokenizer setup, override `download()` in your subclass.

In [ ]:
class SmolLM2_135M(HFModelConfiguration):
    model_name = "HuggingFaceTB/SmolLM2-135M"

## Serve the model

`DeploymentConfig.serve()` builds a vLLM app on Modal, deploys it,
and returns a `ModelDeployment` with the live endpoint URL.
The model weights are downloaded automatically on first deploy.

In [ ]:
from modal_training_gym.common.deployment import DeploymentConfig

model = SmolLM2_135M()
deployment = DeploymentConfig(
    model=model,
    app_name="smollm2-135m-serve",
    served_model_name="smollm2-135m",
).serve()
print(deployment.url)

With the model deployed, we can send it a quick prompt to
verify it's alive.

In [ ]:
response = deployment.generate("What is 2 + 2?")
print(response)

## Define the eval

An `EvalConfig` owns the model-calling loop. You provide:

1. A **dataset** — must expose `load()` returning iterable examples.
2. A **scoring function** (`eval_fn`) — takes `(example, model_response)` and
   returns an `EvalRowResult` with a `score`.
3. A **prompt function** (`prompt_fn`) — takes an example dict and
   returns the prompt string the model sees.

We'll use a tiny slice of GSM8K (grade-school math) and check
whether the model's answer contains the correct number.

In [ ]:
class TinyGSM8K(HuggingFaceDataset):
    hf_repo = "openai/gsm8k"
    hf_config = "main"
    hf_split = "test[:10]"
    input_column = "question"
    output_column = "answer"

In [ ]:
import re

def extract_final_number(text: str) -> str | None:
    matches = re.findall(r"[\d,]+\.?\d*", text.replace(",", ""))
    return matches[-1] if matches else None

def score_gsm8k(example: dict[str, Any], response: str) -> EvalRowResult:
    gold_answer = example["answer"]
    gold_number = extract_final_number(gold_answer.split("####")[-1])
    pred_number = extract_final_number(response)

    correct = gold_number is not None and pred_number == gold_number
    return EvalRowResult(
        score=1.0 if correct else 0.0,
        response=response,
        metadata={"gold": gold_number, "predicted": pred_number},
    )

In [ ]:
def gsm8k_prompt(example: dict[str, Any]) -> str:
    return (
        f"{example['question']}\n\n"
        "Solve step by step. Put your final numeric answer "
        "after ####."
    )

## Run the eval

`evaluate()` iterates over the dataset, calls the model for
each example, scores the response, and returns an `EvalResult`
with per-row detail.

In [ ]:
eval_dataset = TinyGSM8K()

result = EvalConfig(
    deployment=deployment,
    dataset=eval_dataset,
    eval_fn=score_gsm8k,
    prompt_fn=gsm8k_prompt,
).evaluate(debug=True)

print(f"\nAccuracy: {result.accuracy:.1%} ({result.total} examples)")

for i, row in enumerate(result.rows):
    status = "PASS" if row.score >= 1.0 else "FAIL"
    print(f"  [{status}] #{i+1}: gold={row.metadata['gold']} pred={row.metadata['predicted']}")

## What's next

- **Train the model** — plug the same `SmolLM2_135M` into a
  `TrainConfig` with a `SlimeRecipe` (with a `ModelArchitecture`) to GRPO-train it,
  then re-run this eval to measure improvement. See
  [`000_rl_basics`](../../rl/000_rl_basics/000_rl_basics.ipynb).
- **Custom reward** — swap `score_gsm8k` for any function that
  returns `EvalRowResult` — the same signature works as a
  verifiable reward for RL training.